# 02 Source-Lane Dataset EDA

Inspect the PPE v2 input source lanes before splitting. This notebook supports the four-class detector:

- `0 = person`
- `1 = helmet`
- `2 = vest`
- `3 = cleaning_coverall`

It is read-only for dataset contents and does not create generated splits, augmentations, experiments, runs, or weights.


## Purpose

Notebook 01 validates file correctness. Notebook 02 answers whether the source lanes look healthy enough to split:

- How many images and boxes are in each source?
- Does `cleaning_coverall` appear where expected?
- Are factory samples different from open-source samples?
- Are there many tiny boxes that may be difficult for CCTV detection?
- Which images deserve human annotation review before splitting?

To avoid clutter, this notebook saves only compact CSV summaries. Full per-box CSVs and many figure files are intentionally skipped.


## Setup

Find `v2_pipeline`, import EDA helpers, and keep all paths project-relative.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


# Jupyter can run with different working directories. Walk upward until the
# v2_pipeline folder is found so this notebook remains portable.
def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline folder from the current working directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate
    raise RuntimeError(
        "Could not find v2_pipeline root. Run this notebook from inside the repository."
    )


V2_ROOT = find_v2_root(Path.cwd()).resolve()
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from src.analysis.dataset_eda import summarize_source_eda  # noqa: E402

print(f"v2 root: {V2_ROOT}")

v2 root: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline


## Load Configuration

Use YAML for the source folders and class map. This keeps Notebook 02 aligned with Notebook 01 and avoids hardcoded paths or class names.


In [2]:
config_dir = V2_ROOT / "configs"
with (config_dir / "dataset_config.yaml").open("r", encoding="utf-8") as file_handle:
    dataset_config = yaml.safe_load(file_handle)

with (config_dir / "class_names.yaml").open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)

class_names = {
    int(class_id): str(class_name)
    for class_id, class_name in class_config["names"].items()
}

# These are the same source lanes Notebook 01 validates. This notebook only
# reads them; Notebook 03 will create data/generated/splits when we get there.
source_dirs = {
    "open_source": V2_ROOT / dataset_config["input"]["open_source"],
    "factory_source": V2_ROOT / dataset_config["input"]["factory_source"],
    "test_source": V2_ROOT / dataset_config["input"]["test_source"],
}

report_dir = V2_ROOT / dataset_config["reports"]["eda"]
report_dir.mkdir(parents=True, exist_ok=True)

print("Class map:")
for class_id, class_name in class_names.items():
    print(f"- {class_id}: {class_name}")

print("Source lanes:")
for source_name, source_dir in source_dirs.items():
    print(f"- {source_name}: {source_dir}")

Class map:
- 0: person
- 1: helmet
- 2: vest
- 3: cleaning_coverall
Source lanes:
- open_source: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\input\open_source
- factory_source: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\input\factory_source
- test_source: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\input\test_source


## Run Minimal EDA

For each source lane, collect four compact tables:

- source-level image/object summary
- object-level class distribution
- bounding-box size summary by class
- annotation review warnings

The helper intentionally does not write files itself; this cell decides exactly which artifacts are worth saving.


In [3]:
summary_tables = []
class_distribution_tables = []
bbox_size_tables = []
warning_tables = []

for source_name, source_dir in source_dirs.items():
    images_dir = source_dir / "images"
    labels_dir = source_dir / "labels"

    # Skip missing lanes gracefully. This is useful when a source has not been
    # pasted in yet, but it still makes the absence visible in the notebook.
    if not images_dir.exists() or not labels_dir.exists():
        print(f"Skipping {source_name}: missing images/ or labels/ folder.")
        continue

    source_summary, class_distribution, bbox_size_summary, warnings = (
        summarize_source_eda(
            source_name=source_name,
            images_dir=images_dir,
            labels_dir=labels_dir,
            class_names=class_names,
        )
    )
    summary_tables.append(source_summary)
    class_distribution_tables.append(class_distribution)
    bbox_size_tables.append(bbox_size_summary)
    warning_tables.append(warnings)

source_summary_df = (
    pd.concat(summary_tables, ignore_index=True) if summary_tables else pd.DataFrame()
)
class_distribution_df = (
    pd.concat(class_distribution_tables, ignore_index=True)
    if class_distribution_tables
    else pd.DataFrame()
)
bbox_size_summary_df = (
    pd.concat(bbox_size_tables, ignore_index=True)
    if bbox_size_tables
    else pd.DataFrame()
)
eda_warnings_df = (
    pd.concat(warning_tables, ignore_index=True) if warning_tables else pd.DataFrame()
)

print("EDA complete.")
display(source_summary_df)

EDA complete.


,source,num_images,num_labeled_images,num_empty_images,total_objects,mean_objects_per_image,unique_resolutions,num_person,num_helmet,num_vest,num_cleaning_coverall
0,open_source,180,180,0,1751,9.727778,151,722,636,393,0
1,factory_source,637,637,0,4115,6.459969,199,1703,1050,1234,128
2,test_source,88,88,0,352,4.000000,32,158,97,67,30


## Save Compact EDA Reports

Only four CSVs are saved. This keeps the repo easy to navigate while preserving enough information for split planning.


In [4]:
eda_outputs = {
    "source_eda_summary.csv": source_summary_df,
    "source_class_distribution.csv": class_distribution_df,
    "source_bbox_size_summary.csv": bbox_size_summary_df,
    "source_eda_warnings.csv": eda_warnings_df,
}

for filename, frame in eda_outputs.items():
    output_path = report_dir / filename
    frame.to_csv(output_path, index=False)
    print(f"Saved {output_path}")

Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\eda\source_eda_summary.csv
Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\eda\source_class_distribution.csv
Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\eda\source_bbox_size_summary.csv
Saved C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\eda\source_eda_warnings.csv


## Class Distribution Review

Check whether the new `cleaning_coverall` class appears and whether class imbalance is severe. Open-source data can be broader, but factory validation should reflect the target domain.


In [5]:
if class_distribution_df.empty:
    print("No bounding boxes found yet, so class distribution is empty.")
else:
    display(class_distribution_df.sort_values(["source", "class_id"]))

    # Pivot view makes it easier to compare source lanes at a glance.
    class_pivot = class_distribution_df.pivot_table(
        index="source",
        columns="class_name",
        values="object_count",
        fill_value=0,
        aggfunc="sum",
    )
    display(class_pivot)

,source,class_id,class_name,object_count,percentage
3,factory_source,0,person,1703,41.385176
4,factory_source,1,helmet,1050,25.516403
5,factory_source,2,vest,1234,29.987849
6,factory_source,3,cleaning_coverall,128,3.110571
0,open_source,0,person,722,41.233581
1,open_source,1,helmet,636,36.322102
2,open_source,2,vest,393,22.444318
7,test_source,0,person,158,44.886364
8,test_source,1,helmet,97,27.556818
9,test_source,2,vest,67,19.034091


class_name,cleaning_coverall,helmet,person,vest
source,,,,
factory_source,128,1050,1703,1234
open_source,0,636,722,393
test_source,30,97,158,67


## Bounding Box Size Review

Tiny PPE and cleaning-coverall boxes are important because elevated CCTV often makes safety objects small. This summary gives the signal without saving a large per-box artifact.


In [6]:
if bbox_size_summary_df.empty:
    print("No bbox size records available yet.")
else:
    display(bbox_size_summary_df.sort_values(["source", "class_id"]))

,source,class_id,class_name,num_boxes,mean_box_area_norm,median_box_area_norm,mean_box_width_px,mean_box_height_px,tiny_box_count
4,factory_source,0,person,1703,0.018272,0.012383,129.596129,317.613084,7
5,factory_source,1,helmet,1050,0.001244,0.000835,57.095706,47.455803,342
6,factory_source,2,vest,1234,0.008076,0.004606,96.628697,168.340960,94
7,factory_source,3,cleaning_coverall,128,0.019868,0.011419,169.115984,370.954307,0
0,open_source,0,person,722,0.063614,0.034867,145.217764,322.903037,1
1,open_source,1,helmet,636,0.005231,0.002148,68.728971,53.635841,89
2,open_source,2,vest,393,0.030678,0.016068,137.645635,195.196666,3
3,open_source,3,cleaning_coverall,0,0.000000,0.000000,0.000000,0.000000,0
8,test_source,0,person,158,0.017364,0.009243,77.975770,201.445978,2
9,test_source,1,helmet,97,0.001058,0.000510,33.446114,26.416773,47


## Annotation Warning Review

Warnings are prompts for human review, not automatic failures. They are useful before splitting because annotation gaps can otherwise leak into every downstream experiment.


In [7]:
if eda_warnings_df.empty:
    print("No EDA warning rows found.")
else:
    warning_counts = (
        eda_warnings_df.groupby(["source", "warning_type"], as_index=False)
        .size()
        .rename(columns={"size": "count"})
        .sort_values(["source", "count"], ascending=[True, False])
    )
    display(warning_counts)
    display(eda_warnings_df.head(50))

,source,warning_type,count
3,factory_source,very_tiny_box,443
1,factory_source,person_without_helmet,181
0,factory_source,many_objects,14
2,factory_source,person_without_role_uniform,14
8,open_source,very_tiny_box,93
6,open_source,person_without_role_uniform,60
4,open_source,many_objects,10
5,open_source,person_without_helmet,8
7,open_source,very_large_box,2
12,test_source,very_tiny_box,65


,source,image_name,warning_type,details
0,open_source,ppe_0008.jpg,person_without_helmet,"4 person boxes, 0 helmet boxes"
1,open_source,ppe_0018.jpg,person_without_role_uniform,"2 person boxes, 0 vest boxes, 0 cleaning_cover..."
2,open_source,ppe_0037.jpg,person_without_role_uniform,"5 person boxes, 0 vest boxes, 0 cleaning_cover..."
3,open_source,ppe_0055.jpg,person_without_role_uniform,"1 person boxes, 0 vest boxes, 0 cleaning_cover..."
4,open_source,ppe_0059.jpg,person_without_role_uniform,"2 person boxes, 0 vest boxes, 0 cleaning_cover..."
5,open_source,ppe_0101.jpg,person_without_role_uniform,"4 person boxes, 0 vest boxes, 0 cleaning_cover..."
6,open_source,ppe_0128.jpg,person_without_role_uniform,"4 person boxes, 0 vest boxes, 0 cleaning_cover..."
7,open_source,ppe_0191.jpg,person_without_helmet,"3 person boxes, 0 helmet boxes"
8,open_source,ppe_0296.jpg,person_without_role_uniform,"2 person boxes, 0 vest boxes, 0 cleaning_cover..."
9,open_source,ppe_0325.jpg,person_without_role_uniform,"8 person boxes, 0 vest boxes, 0 cleaning_cover..."


## Source Policy Reminder

Notebook 02 does not split data. Notebook 03 should apply this policy:

- `train`: valid `open_source` samples plus the training portion of valid `factory_source` samples.
- `val`: validation portion of valid `factory_source` samples only.
- `test`: valid `test_source` samples only.

The test source may be inspected here for data readiness, but it must not drive architecture, augmentation, hyperparameter, threshold, or ablation choices.


## Final Checklist

Before Notebook 03:

- `source_eda_summary.csv` was reviewed.
- `source_class_distribution.csv` includes all expected classes, especially `cleaning_coverall` if present in the dataset.
- `source_bbox_size_summary.csv` was checked for many tiny boxes.
- `source_eda_warnings.csv` was reviewed for annotation issues.
- No generated split, augmentation, experiment dataset, training run, or model artifact was created by this notebook.
